# Etapa 1 — Exploración del dataset

Carga del PhiUSIIL Phishing URL Dataset y chequeos básicos. Solo vamos a usar la columna `URL`.

In [ ]:
import pandas as pd
import tldextract

RUTA_CSV = "PhiUSIIL_Phishing_URL_Dataset.csv"

# El CSV viene con BOM UTF-8; con "utf-8-sig" la primera columna queda como "FILENAME"
df = pd.read_csv(RUTA_CSV, encoding="utf-8-sig")

# Chequeo: label debería tener solo los valores 0 y 1
valores_label = set(df["label"].unique())
if valores_label != {0, 1}:
    print(f"ATENCIÓN: 'label' tiene valores inesperados: {valores_label}")

# En el dataset original label = 1 es LEGÍTIMA y label = 0 es PHISHING.
# Invertimos para trabajar siempre con phishing = 1.
df["phishing"] = 1 - df["label"]

In [ ]:
# Forma del dataset
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

In [ ]:
# Columnas disponibles (solo usaremos URL; el resto son features precalculadas que ignoramos)
print(list(df.columns))

In [ ]:
# Balance de clases
nombres_clase = {0: "legítima", 1: "phishing"}
balance = pd.DataFrame({
    "cantidad": df["phishing"].value_counts(),
    "proporción": df["phishing"].value_counts(normalize=True).round(4),
}).rename(index=nombres_clase)
balance

In [ ]:
# 5 URLs de ejemplo de cada clase
for clase, nombre in nombres_clase.items():
    print(f"--- {nombre} ---")
    ejemplos = df.loc[df["phishing"] == clase, "URL"].sample(5, random_state=42)
    for url in ejemplos:
        print(url)
    print()

In [ ]:
# Dominios registrables únicos (misma definición que usa features.py)
from features import dominio_registrable

dominios = df["URL"].map(dominio_registrable)
print(f"Dominios registrables únicos (total): {dominios.nunique():,}")
for clase, nombre in nombres_clase.items():
    print(f"Dominios registrables únicos ({nombre}): {dominios[df['phishing'] == clase].nunique():,}")

# Etapa 2 — Prueba de extract_features

Probamos la función de `features.py` con algunas URLs a mano (todavía no con el dataset completo).

In [ ]:
from features import extract_features

urls_prueba = [
    "https://www.bna.com.ar",
    "http://bna-homebanking-verificar.xyz/login",
    "https://www.afip.gob.ar/",                            # sufijo .gob.ar
    "http://192.168.0.1:8080/paypal/login.php?id=1&t=2",   # IP, puerto, marca y parámetros
    "https://bit.ly/3xYz12",                               # acortador
    "https://storage.googleapis.com/algo",                 # dominio oficial de Google
    "http://paypal-login-seguro.com",                      # marca fuera de su dominio
]

# Una columna por URL para compararlas lado a lado
pd.DataFrame([extract_features(u) for u in urls_prueba], index=urls_prueba).T

**Nota: posible artefacto del dataset.** En PhiUSIIL muchas URLs legítimas son solo la página de inicio (`https://www.dominio.com`) y el phishing suele tener path. Por eso `longitud_path`, `profundidad_path`, `usa_https` y `longitud_url` pueden separar las clases por cómo se armó el dataset y no por una señal real de phishing. Si más adelante las métricas dan casi perfectas, es la primera causa a revisar.

# Etapa 3 — Features del dataset completo

Aplicamos `extract_features` a todas las URLs y guardamos el resultado en `data/features.parquet`.

In [ ]:
import time

import numpy as np

from features import dominio_registrable, extract_features

inicio = time.perf_counter()
X = pd.DataFrame([extract_features(u) for u in df["URL"]])
y = df["phishing"].reset_index(drop=True)
dominio = df["URL"].map(dominio_registrable).reset_index(drop=True)
duracion = time.perf_counter() - inicio

print(f"Forma de X: {X.shape}")
print(f"Tiempo: {duracion:.1f} s ({len(X) / duracion:,.0f} URLs/s)")

In [ ]:
# Controles de calidad (solo se muestran; no se elimina nada)
nulos = X.isna().sum()
print(f"Valores nulos: {nulos.sum()}")
if nulos.sum():
    print(nulos[nulos > 0])

infinitos = pd.Series(np.isinf(X.to_numpy(dtype=float)).sum(axis=0), index=X.columns)
print(f"Valores infinitos: {infinitos.sum()}")
if infinitos.sum():
    print(infinitos[infinitos > 0])

print(f"Filas con dominio vacío: {(dominio == '').sum()}")

urls = df["URL"].reset_index(drop=True)
print(f"Filas con URL duplicada: {urls.duplicated().sum():,}")
print(f"URLs distintas que se repiten: {urls[urls.duplicated(keep=False)].nunique():,}")
etiquetas_por_url = df.groupby("URL")["phishing"].nunique()
print(f"URLs duplicadas con etiquetas distintas: {(etiquetas_por_url > 1).sum():,}")

**Columnas que NO son features** en `data/features.parquet`:
- `URL`: se guarda solo para inspeccionar los errores del modelo en la etapa 4.
- `phishing`: el target (1 = phishing).
- `dominio`: dominio registrable, se usa como grupo en `GroupShuffleSplit`.

Al entrenar, hay que excluir estas tres columnas de `X`.

In [ ]:
from pathlib import Path

RUTA_FEATURES = Path("data") / "features.parquet"
RUTA_FEATURES.parent.mkdir(exist_ok=True)

df_features = X.assign(URL=urls, phishing=y, dominio=dominio)
df_features.to_parquet(RUTA_FEATURES, index=False)

print(f"Guardado en {RUTA_FEATURES}")
print(f"Forma: {df_features.shape}")
print(f"Tamaño: {RUTA_FEATURES.stat().st_size / 1e6:.1f} MB")

# Etapa 4 — Entrenamiento y evaluación

Cargamos las features guardadas, dividimos por dominio registrable, comparamos dos modelos y medimos cuánto dependen del atajo del dataset. El modelo todavía no se guarda.

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit

from features import extract_features

datos = pd.read_parquet("data/features.parquet")

# URL, phishing y dominio no son features
NO_FEATURES = ["URL", "phishing", "dominio"]
X = datos.drop(columns=NO_FEATURES)
y = datos["phishing"]
grupos = datos["dominio"]

assert X.shape[1] == 23, f"Se esperaban 23 features y hay {X.shape[1]}"
print(f"Forma de X: {X.shape}")

In [ ]:
# Dominios antes de dividir
por_dominio = pd.crosstab(grupos, y).rename(columns={0: "legítima", 1: "phishing"})
por_dominio["total"] = por_dominio["legítima"] + por_dominio["phishing"]
por_dominio = por_dominio.sort_values("total", ascending=False)

print("15 dominios con más URLs:")
display(por_dominio.head(15))

ambas_clases = por_dominio[(por_dominio["legítima"] > 0) & (por_dominio["phishing"] > 0)]
print(f"Dominios con URLs de ambas clases: {len(ambas_clases)}")
display(ambas_clases.head(20))

In [ ]:
# División por dominio registrable (el 20% es de dominios, no de filas)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(gss.split(X, y, groups=grupos))

X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]
dom_train, dom_test = grupos.iloc[idx_train], grupos.iloc[idx_test]

compartidos = set(dom_train) & set(dom_test)
print(f"Dominios compartidos entre train y test: {len(compartidos)}")
assert not compartidos, "Hay dominios en ambos conjuntos"

print(f"Filas train: {len(X_train):,} ({len(X_train) / len(X):.1%})")
print(f"Filas test:  {len(X_test):,} ({len(X_test) / len(X):.1%})")

nombres_clase = {0: "legítima", 1: "phishing"}
balance_split = pd.DataFrame({
    "train": y_train.value_counts(),
    "train %": y_train.value_counts(normalize=True).round(4),
    "test": y_test.value_counts(),
    "test %": y_test.value_counts(normalize=True).round(4),
}).rename(index=nombres_clase)
display(balance_split)

In [ ]:
def evaluar(nombre, modelo, X_te, y_te):
    """Imprime reporte, matriz de confusión, ROC-AUC y FPR; devuelve las métricas."""
    proba = modelo.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)

    print(f"===== {nombre} =====")
    print(classification_report(y_te, pred, target_names=["legítima", "phishing"], digits=4))

    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    matriz = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["real legítima", "real phishing"],
        columns=["pred. legítima", "pred. phishing"],
    )
    print(matriz, "\n")

    metricas = {
        "modelo": nombre,
        "roc_auc": roc_auc_score(y_te, proba),
        "recall_phishing": recall_score(y_te, pred),
        "precision_phishing": precision_score(y_te, pred),
        "fpr": fp / (fp + tn),
        "accuracy": accuracy_score(y_te, pred),
        "proba": proba,
    }
    print(f"ROC-AUC: {metricas['roc_auc']:.4f} | tasa de falsos positivos: {metricas['fpr']:.4f}\n")
    return metricas


def tabla(resultados):
    """Tabla comparativa sin la columna de probabilidades."""
    return pd.DataFrame([{k: v for k, v in r.items() if k != "proba"} for r in resultados]).set_index("modelo").round(4)

In [ ]:
# Entrenamiento de los dos modelos con las 23 features
modelos = {
    "RandomForest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}

resultados = []
entrenados = {}
for nombre, modelo in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    print(f"{nombre}: entrenado en {time.perf_counter() - inicio:.1f} s")
    entrenados[nombre] = modelo
    resultados.append(evaluar(nombre, modelo, X_test, y_test))

In [ ]:
# Comparación y elección del mejor (ROC-AUC; si empatan a 4 decimales, menor FPR)
display(tabla(resultados))

mejor_res = min(resultados, key=lambda r: (-round(r["roc_auc"], 4), r["fpr"]))
nombre_mejor = mejor_res["modelo"]
mejor = entrenados[nombre_mejor]
print(f"Mejor modelo: {nombre_mejor}")

sospechosos = [r["modelo"] for r in resultados if r["roc_auc"] >= 0.99 or r["accuracy"] >= 0.99]
if sospechosos:
    print(
        f"\nATENCIÓN: métricas sospechosamente altas en {', '.join(sospechosos)}.\n"
        "Causas probables: atajo https/www/path (las legítimas son casi siempre la home con HTTPS),\n"
        "hostings gratuitos que concentran phishing y la forma en que se armó el dataset."
    )

In [ ]:
# Prueba del atajo del dataset
ATAJO = ["longitud_path", "profundidad_path", "usa_https", "longitud_url"]
TRIVIAL = ["usa_https", "profundidad_path"]

sin_atajo = clone(mejor).fit(X_train.drop(columns=ATAJO), y_train)
res_sin_atajo = evaluar(f"{nombre_mejor} sin atajo", sin_atajo, X_test.drop(columns=ATAJO), y_test)

# Modelo trivial de diagnóstico: solo HTTPS y profundidad del path
trivial = clone(mejor).fit(X_train[TRIVIAL], y_train)
res_trivial = evaluar(f"{nombre_mejor} trivial (solo {' + '.join(TRIVIAL)})", trivial, X_test[TRIVIAL], y_test)

comparacion_atajo = tabla([mejor_res, res_sin_atajo, res_trivial])
display(comparacion_atajo)
print("Diferencia sin atajo - completo:")
display((comparacion_atajo.iloc[1] - comparacion_atajo.iloc[0]).round(4).to_frame("diferencia"))

In [ ]:
# Punto de operación con FPR <= 1% (en una extensión los falsos positivos son muy costosos)
def recall_con_fpr_max(y_te, proba, fpr_max=0.01):
    """Mayor recall alcanzable con FPR <= fpr_max; devuelve (recall, umbral, fpr)."""
    fpr, tpr, umbrales = roc_curve(y_te, proba)
    validos = np.where(fpr <= fpr_max)[0]
    i = validos[np.argmax(tpr[validos])]
    return tpr[i], umbrales[i], fpr[i]


umbrales_op = {}
filas = []
for res in [mejor_res, res_sin_atajo]:
    recall, umbral, fpr_real = recall_con_fpr_max(y_test, res["proba"])
    umbrales_op[res["modelo"]] = umbral
    filas.append({"modelo": res["modelo"], "recall_phishing": recall, "umbral": umbral, "fpr_real": fpr_real})

print("Recall de phishing con FPR <= 1%:")
display(pd.DataFrame(filas).set_index("modelo").round(4))

In [ ]:
# URLs de prueba: las 7 de la etapa 2 + 5 legítimas con path (navegación real)
urls_prueba = {
    "https://www.bna.com.ar": "legítima",
    "http://bna-homebanking-verificar.xyz/login": "phishing",
    "https://www.afip.gob.ar/": "legítima",
    "http://192.168.0.1:8080/paypal/login.php?id=1&t=2": "phishing",
    "https://bit.ly/3xYz12": "acortador",
    "https://storage.googleapis.com/algo": "legítima",
    "http://paypal-login-seguro.com": "phishing",
    "https://www.mercadolibre.com.ar/ofertas": "legítima",
    "https://www.argentina.gob.ar/anses/jubilaciones": "legítima",
    "https://github.com/scikit-learn/scikit-learn/issues": "legítima",
    "https://www.bna.com.ar/Personas/Prestamos": "legítima",
    "https://es.wikipedia.org/wiki/Phishing": "legítima",
}

X_urls = pd.DataFrame([extract_features(u) for u in urls_prueba])[X.columns]
versiones = {
    "completo": (mejor, X_urls, umbrales_op[mejor_res["modelo"]]),
    "sin atajo": (sin_atajo, X_urls.drop(columns=ATAJO), umbrales_op[res_sin_atajo["modelo"]]),
}

prueba = pd.DataFrame({"esperado": list(urls_prueba.values())}, index=list(urls_prueba))
es_legitima = prueba["esperado"] == "legítima"
for nombre, (modelo, X_v, umbral) in versiones.items():
    p = modelo.predict_proba(X_v)[:, 1]
    prueba[f"p_phishing ({nombre})"] = p.round(3)
    # Marca falsos positivos en URLs legítimas: con umbral 0.5 y con el umbral de FPR <= 1%
    marcas = []
    for legit, prob in zip(es_legitima, p):
        m = [etiqueta for etiqueta, u in [("0.5", 0.5), ("FPR≤1%", umbral)] if legit and prob >= u]
        marcas.append(f"FP ({', '.join(m)})" if m else "")
    prueba[f"FP ({nombre})"] = marcas
    print(f"Umbral FPR ≤ 1% ({nombre}): {umbral:.3f}")

display(prueba)

## Conclusiones de la etapa 4

**Las métricas del modelo completo son sospechosamente perfectas y no representan el uso real.** HistGradientBoosting (el mejor) da ROC-AUC 0.998, recall de phishing 0.990 y FPR 0.10% en test. Pero:

- **El modelo trivial con solo `usa_https` y `profundidad_path` ya logra ROC-AUC 0.936 con FPR 0%.** Ninguna URL legítima del test tiene path o usa HTTP. El dataset está armado así: las legítimas son homes `https://www.dominio.tld`.
- **En las URLs de prueba, el modelo completo marca como phishing (p = 1.0) a TODAS las legítimas con path**, e incluso a `https://www.afip.gob.ar/` solo por la `/` final. Son falsos positivos: afip.gob.ar/, storage.googleapis.com, mercadolibre.com.ar/ofertas, argentina.gob.ar/anses/jubilaciones, github.com/…/issues, bna.com.ar/Personas/Prestamos y es.wikipedia.org/wiki/Phishing. En una extensión, eso significa alertar en casi cualquier página que visite el usuario.

**Versión sin ATAJO** (sin `longitud_path`, `profundidad_path`, `usa_https` ni `longitud_url`):
- En test baja a ROC-AUC 0.865, recall 0.684 y FPR 3.65% con umbral 0.5. Con **FPR ≤ 1%**, el recall es **0.596** con umbral **0.80**.
- En las URLs de prueba se comporta de forma razonable: los 3 phishing y el acortador dan ~1.0, y las legítimas quedan entre 0.06 y 0.19. **Única excepción: `https://github.com/scikit-learn/scikit-learn/issues` da 1.0, un falso positivo** incluso con el umbral de 0.80.
- No elimina el atajo del todo: `cant_puntos`, `cant_digitos`, `cant_especiales`, `proporcion_digitos` y `cant_parametros` también se calculan sobre la URL completa, así que el path sigue influyendo de forma indirecta. Las métricas de test de esta versión probablemente siguen siendo optimistas.

**Recomendación:** usar la **versión sin ATAJO con umbral ~0.80** (punto de operación con FPR ≤ 1%). Tiene menos recall en el papel, pero el modelo completo es inutilizable en navegación real porque confunde "tiene path" con "es phishing". El recall que falta lo pueden compensar las reglas y el LLM del backend. Queda pendiente para una etapa futura: calcular esos conteos solo sobre el hostname y revisar el falso positivo de github.

Nota: con la división por dominio, el test quedó con 64% de legítimas contra 56% en train, porque hostings grandes que son 100% phishing (web.app, firebaseapp.com, repl.co…) caen enteros de un lado.

# Etapa 4b — Modelo basado solo en el dominio

Como ninguna URL legítima del dataset tiene path, cualquier feature calculada sobre la URL completa es un atajo. Acá usamos `extract_features_dominio`, que mira solo el hostname (sin esquema, path, query ni puerto) y además saca el prefijo `www.`.

In [ ]:
from features import extract_features_dominio

# Prueba con las 12 URLs
pd.DataFrame([extract_features_dominio(u) for u in urls_prueba], index=list(urls_prueba)).T

In [ ]:
# Recalcular las features del dataset con la versión de dominio
from pathlib import Path

base = pd.read_parquet("data/features.parquet", columns=["URL", "phishing", "dominio"])

inicio = time.perf_counter()
X_dom = pd.DataFrame([extract_features_dominio(u) for u in base["URL"]])
duracion = time.perf_counter() - inicio
print(f"Forma de X_dom: {X_dom.shape}")
print(f"Tiempo: {duracion:.1f} s")
print(f"Valores nulos: {X_dom.isna().sum().sum()}")
print(f"Valores infinitos: {np.isinf(X_dom.to_numpy(dtype=float)).sum()}")

RUTA_FEATURES_DOMINIO = Path("data") / "features_dominio.parquet"
X_dom.assign(URL=base["URL"], phishing=base["phishing"], dominio=base["dominio"]).to_parquet(
    RUTA_FEATURES_DOMINIO, index=False
)
print(f"Guardado en {RUTA_FEATURES_DOMINIO} ({RUTA_FEATURES_DOMINIO.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Diagnóstico del atajo "www" sobre el hostname ORIGINAL (antes de normalizar)
tiene_www = base["URL"].str.lower().str.match(r"^(?:[a-z][a-z0-9+.-]*://)?www\.")
print("Proporción de hostnames que empiezan con 'www.':")
display(tiene_www.groupby(base["phishing"]).mean().rename(index=nombres_clase).round(4).to_frame("con www."))

In [ ]:
# Mismo split por dominio que la etapa 4
assert (base["dominio"].to_numpy() == grupos.to_numpy()).all(), "El orden de filas no coincide con la etapa 4"
assert not set(base["dominio"].iloc[idx_train]) & set(base["dominio"].iloc[idx_test])

Xd_train, Xd_test = X_dom.iloc[idx_train], X_dom.iloc[idx_test]

modelo_dom = HistGradientBoostingClassifier(random_state=42).fit(Xd_train, y_train)
res_dom = evaluar("HGB dominio", modelo_dom, Xd_test, y_test)

filas = []
for res in [res_sin_atajo, res_dom]:
    recall_op, umbral, fpr_real = recall_con_fpr_max(y_test, res["proba"])
    umbrales_op[res["modelo"]] = umbral
    filas.append({**{k: v for k, v in res.items() if k != "proba"},
                  "recall_fpr1%": recall_op, "umbral_fpr1%": umbral})

print("Comparación con la versión sin atajo de la etapa 4:")
display(pd.DataFrame(filas).set_index("modelo").round(4))

In [ ]:
# 12 URLs: modelo de dominio vs. versión sin atajo de la etapa 4
X_urls_dom = pd.DataFrame([extract_features_dominio(u) for u in urls_prueba])[X_dom.columns]
versiones_4b = {
    "sin atajo": (sin_atajo, X_urls.drop(columns=ATAJO), umbrales_op[res_sin_atajo["modelo"]]),
    "dominio": (modelo_dom, X_urls_dom, umbrales_op[res_dom["modelo"]]),
}

prueba_4b = pd.DataFrame({"esperado": list(urls_prueba.values())}, index=list(urls_prueba))
es_legitima = prueba_4b["esperado"] == "legítima"
for nombre, (modelo, X_v, umbral) in versiones_4b.items():
    p = modelo.predict_proba(X_v)[:, 1]
    prueba_4b[f"p_phishing ({nombre})"] = p.round(3)
    marcas = []
    for legit, prob in zip(es_legitima, p):
        m = [etiqueta for etiqueta, u in [("0.5", 0.5), ("FPR≤1%", umbral)] if legit and prob >= u]
        marcas.append(f"FP ({', '.join(m)})" if m else "")
    prueba_4b[f"FP ({nombre})"] = marcas
    print(f"Umbral FPR ≤ 1% ({nombre}): {umbral:.3f}")

display(prueba_4b)

In [ ]:
# ¿Qué feature causa cada legítima marcada como FP por el modelo de dominio?
# Reemplazamos cada feature, una por vez, por la mediana de las legítimas de train
# y medimos cuánto baja p_phishing.
mediana_legit = Xd_train[y_train.to_numpy() == 0].median()
fp_dominio = prueba_4b.index[prueba_4b["FP (dominio)"] != ""]

if len(fp_dominio) == 0:
    print("Ninguna URL legítima de prueba da alta con el modelo de dominio.")
for url in fp_dominio:
    fila = X_urls_dom.iloc[[list(urls_prueba).index(url)]]
    p_base = modelo_dom.predict_proba(fila)[0, 1]
    impactos = []
    for col in X_dom.columns:
        modificada = fila.copy()
        modificada[col] = mediana_legit[col]
        impactos.append({
            "feature": col,
            "valor_url": fila[col].iloc[0],
            "mediana_legítimas": mediana_legit[col],
            "baja_p": p_base - modelo_dom.predict_proba(modificada)[0, 1],
        })
    print(f"\n{url}  (p_phishing = {p_base:.3f})")
    display(pd.DataFrame(impactos).sort_values("baja_p", ascending=False).head(5).round(3).set_index("feature"))

## Conclusiones de la etapa 4b

**Atajo de `www` confirmado:** el 100% de las legítimas del dataset empiezan con `www.`, contra el 41% del phishing. Por eso normalizarlo era necesario.

**En test, el modelo de dominio rinde menos que la versión sin atajo**, pero esa versión sigue viendo el path de forma indirecta y el de dominio no:

| | ROC-AUC | recall (0.5) | FPR (0.5) | recall con FPR ≤ 1% | umbral |
|---|---|---|---|---|---|
| sin atajo (etapa 4) | 0.865 | 0.684 | 3.65% | 0.596 | 0.80 |
| dominio (4b) | 0.820 | 0.572 | 3.47% | 0.485 | 0.84 |

Las métricas ya no dan sospechosamente perfectas. Es un resultado más honesto de lo que se puede saber mirando solo el hostname.

**En las 12 URLs:**
- Detecta los 3 phishing y el acortador (p ≥ 0.98). Todas las legítimas de dominio "plano" quedan bajas: bna, afip, mercadolibre, argentina.gob.ar y github (0.10–0.34).
- **Se corrige el falso positivo de github** (1.0 → 0.20). Venía de las features de la URL completa.
- **Falsos positivos nuevos:**
  - `https://storage.googleapis.com/algo` da **0.88** y supera el umbral de 0.84;
  - `https://es.wikipedia.org/wiki/Phishing` da **0.63**; supera 0.5 pero no el umbral.

**Qué feature los causa:** `cant_subdominios`. Reemplazarla por la mediana de las legítimas (0) baja p en 0.71 (googleapis) y 0.47 (wikipedia); en googleapis también influye `longitud_dominio`, con −0.27. Es un **atajo residual del dataset**: después de sacar `www`, solo el 2.9% de las legítimas tiene algún subdominio, contra el 54% del phishing. Las legítimas de PhiUSIIL son homes `www.dominio.tld`, pero en la navegación real los subdominios legítimos son comunes (es.wikipedia.org, docs.google.com, storage.googleapis.com).

**Recomendación:** preferir el **modelo de dominio con umbral ~0.84**. Rinde menos en test, pero es el único que no penaliza el path, que es lo más frecuente al navegar. Su debilidad conocida son los subdominios legítimos. Hay dos opciones para una etapa siguiente: sacar o acotar `cant_subdominios` y medir el costo, o completar el dataset con URLs legítimas que tengan subdominios.